# 09 — V2 feature stability & frozen comparison

Two questions, one notebook:

1. **Are the V2 LLM count features stable?** Re-run the *identical* V2 prompt with a second,
   independent model (`qwen3.6-35b` vs the original `qwen3.5:122b`) and measure per-feature
   agreement (Spearman ρ, ICC(2,1), exact-match).
2. **Do V2 features add information beyond simple surface features?** Compare V2 against
   transcript length, surface statistics, V1 categorical features and a TF-IDF ceiling under
   the same repeated-CV protocol as notebooks 04–08, now with **paired** bootstrap tests.

Automatic gates (no eyeballing — this is for the methods section):

* **Degeneracy gate** — a count feature is dropped if `zero_share > 0.70` on the labelled set
  in *either* run.
* **Stability flag** — a feature is "stable" if Spearman ρ(run A, run B) ≥ 0.60. Reported for
  every feature; a stable-only sensitivity arm is included.

The pipeline (features → preprocessing → classifier → params) is **frozen** to disk at the end.
The 61-case test set is **not** touched here.

## Block 0 — configuration

In [1]:
import os, json, time, hashlib, datetime, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# .env lives at the repo root, one level above notebooks_preliminary/
for p in (Path("../.env"), Path(".env")):
    if p.exists():
        load_dotenv(p); break

warnings.filterwarnings("ignore")

BASE_URL = "https://litellm.vse.cz/v1"
API_KEY  = os.environ["LOCAL_OPENAI_API_KEY"]

MODEL_A = "qwen3.5:122b"     # original V2 run  -> fileDataset/outputs/v2_raw.jsonl  (notebook 08)
MODEL_B = "qwen3.6-35b"      # second model for the stability check -> v2b_raw.jsonl (this notebook)

TEMPERATURE = 0.0
MAX_TOKENS  = 4096
TIMEOUT_S   = 600

RUN_LLM = False              # both v2_raw.jsonl + v2b_raw.jsonl are cached -> analysis-only re-run
                             # (set True to re-extract with MODEL_B from scratch)
N_CALIB = 5
SEED    = 42

DATA = Path("fileDataset")
OUT  = DATA / "outputs"
OUT.mkdir(exist_ok=True, parents=True)

RAW_A      = OUT / "v2_raw.jsonl"                 # produced by notebook 08 (MODEL_A)
RAW_B      = OUT / "v2b_raw.jsonl"                # produced here (MODEL_B), appended live
META_B     = OUT / "v2b_run_metadata.json"
FROZEN_DIR = OUT / "frozen_pipeline_v2"
FROZEN_DIR.mkdir(exist_ok=True, parents=True)

# ---- automatic gate thresholds ------------------------------------------------
ZERO_SHARE_MAX    = 0.70     # degeneracy gate
STABILITY_MIN_RHO = 0.60     # "stable" if Spearman rho(A, B) >= this

EXPECTED_PROMPT_SHA = "f52aae5d939d620e"   # from run_metadata.json — must match to compare runs

assert RAW_A.exists(), f"{RAW_A} not found — run notebook 08 first"
print("MODEL_A :", MODEL_A, "| cached  :", RAW_A, "|", RAW_A.exists())
print("MODEL_B :", MODEL_B, "| target  :", RAW_B, "|", RAW_B.exists())
print("RUN_LLM :", RUN_LLM)

MODEL_A : qwen3.5:122b | cached  : fileDataset/outputs/v2_raw.jsonl | True
MODEL_B : qwen3.6-35b | target  : fileDataset/outputs/v2b_raw.jsonl | True
RUN_LLM : False


## Block 1 — the V2 prompt (byte-identical to notebook 08)

The stability comparison is only meaningful if the prompt is exactly the same. The SHA is
asserted against the value recorded in `run_metadata.json`.

In [2]:
V2_PROMPT = r"""You are annotating a transcript of spontaneous Czech speech. A person was shown a drawing of a
lakeshore scene and asked to describe it aloud. The text is an automatic transcription.

Your job is to COUNT observable linguistic events and QUOTE the evidence for each count.
You are an annotator, not an evaluator.

RULES
- Quote evidence verbatim from the transcript, in Czech. Never translate or paraphrase.
- If a category has no instances, return an empty list. Empty is a valid answer.
- Never infer anything about the speaker: not their health, ability, intelligence, age,
  education or state of mind. Describe the language, never the person.
- Do not compare this speaker to anyone else or to any norm.
- Count occurrences, not impressions. Two hedges in one sentence are two entries.
- Return only JSON.

CATEGORIES
1. named_entities - distinct objects, creatures or people explicitly named. Lemmatise and list
   each distinct entity exactly once (a set, not a list of mentions).
2. specific_action_verbs - verbs naming a particular manner of action. List every occurrence.
3. generic_verbs - verbs of bare existence, possession, location or unspecified movement.
4. complete_propositions - integer count of clauses with an explicit subject and predicate
   plus at least one further argument or adjunct.
5. locative_expressions - phrases placing something somewhere. List every occurrence.
6. regions_referenced - distinct regions among "water", "land", "sky".
7. hedge_spans - expressions of uncertainty. List every occurrence.
8. deictic_spans - references such as "there", "that thing" that substitute for naming.
9. metacomment_spans - remarks about the speaker's own describing, remembering, or the task.
10. repeated_content_lemmas - content words used more than once, with counts.
11. self_corrections - integer count.
12. diminutive_or_affective_forms - diminutive or affectionate noun forms.
13. quantity_expressions - numerals or quantifiers applied to things in the scene.

Return exactly this JSON and nothing else:
{
  "named_entities": [],
  "specific_action_verbs": [],
  "generic_verbs": [],
  "complete_propositions": 0,
  "locative_expressions": [],
  "regions_referenced": [],
  "hedge_spans": [],
  "deictic_spans": [],
  "metacomment_spans": [],
  "repeated_content_lemmas": [{"lemma": "", "count": 0}],
  "self_corrections": 0,
  "diminutive_or_affective_forms": [],
  "quantity_expressions": []
}"""

PROMPT_SHA = hashlib.sha256(V2_PROMPT.encode()).hexdigest()[:16]
print(f"prompt sha256[:16] = {PROMPT_SHA}  ({len(V2_PROMPT)} chars)")
assert PROMPT_SHA == EXPECTED_PROMPT_SHA, (
    f"prompt SHA {PROMPT_SHA} != {EXPECTED_PROMPT_SHA} recorded for run A — "
    "the two runs would not be comparable")

for banned in ["dementia", "alzheimer", "cognitive", "impair", "patient", "diagnos",
               "control group", "demence", "pacient"]:
    assert banned not in V2_PROMPT.lower(), f"prompt leaks domain term: {banned}"
print("prompt matches run A, domain-blind check: OK")

prompt sha256[:16] = f52aae5d939d620e  (2428 chars)
prompt matches run A, domain-blind check: OK


## Block 2 — load transcripts + surface statistics

`train/negative` (0), `train/positive` (1), `overview/` (unlabelled). `test/` is never read.

In [3]:
if not DATA.exists():
    raise FileNotFoundError(f"dataset not found at {DATA}")

SPLITS = [("train/negative", 0), ("train/positive", 1), ("overview", np.nan)]

def read_split(folder: Path, label):
    return [{"file": f.name, "label": label,
             "text": f.read_text(encoding="utf-8", errors="replace").strip()}
            for f in sorted(folder.glob("*.txt"))]

docs = pd.DataFrame([r for sub, lab in SPLITS for r in read_split(DATA / sub, lab)])
assert docs.file.is_unique and not docs.empty

WORD = re.compile(r"\w+", re.UNICODE)
docs["n_word"]  = docs.text.str.split().str.len().clip(lower=1)
docs["n_char"]  = docs.text.str.len()
docs["n_type"]  = docs.text.str.lower().apply(lambda t: len(set(WORD.findall(t))))
docs["ttr"]     = docs.n_type / docs.n_word
docs["n_sent"]  = docs.text.apply(lambda t: max(1, len(re.findall(r"[.!?]+", t))))
docs["mlu"]     = docs.n_word / docs.n_sent
docs["n_comma"] = docs.text.str.count(",")
SURFACE = ["n_word", "n_type", "ttr", "n_sent", "mlu", "n_comma", "n_char"]

print(f"{len(docs)} docs -> {docs.label.notna().sum()} labelled + {docs.label.isna().sum()} unlabelled")
print(docs.loc[docs.label.notna(), "label"].astype(int)
          .map({0: "negative", 1: "positive"}).value_counts().to_string())

327 docs -> 241 labelled + 86 unlabelled
label
negative    171
positive     70


## Block 3 — provider (identical `StreamingLocalProvider` as notebook 08)

Streaming + `think:false` + retries on transient failures. Only the target model changes.

In [4]:
import openai
from openai import BadRequestError
from llm_feature_gen.providers.local_provider import LocalProvider

THINK_TAG = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)


class StreamingLocalProvider(LocalProvider):
    """LocalProvider + streaming + think-off + retries on transient failures."""

    def __init__(self, *a, stream: bool = True, **kw):
        super().__init__(*a, **kw)
        self.stream = stream
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key,
                                    timeout=TIMEOUT_S)
        self._think_supported = True

    def _raw_call(self, model, messages, json_mode):
        kw = dict(model=model, messages=messages,
                  temperature=self.temperature, max_tokens=self.max_tokens)
        if json_mode:
            kw["response_format"] = {"type": "json_object"}
        if self._think_supported:
            kw["extra_body"] = {"think": False}

        if not self.stream:
            return self.client.chat.completions.create(**kw).choices[0].message.content or ""

        parts = []
        for ev in self.client.chat.completions.create(stream=True, **kw):
            if ev.choices and ev.choices[0].delta and ev.choices[0].delta.content:
                parts.append(ev.choices[0].delta.content)
        return "".join(parts)

    def _chat_json(self, deployment_name, system_prompt, user_content, json_mode=False):
        if json_mode and "JSON" not in system_prompt:
            system_prompt += " Respond in strict JSON format."
        messages = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}]

        last, backoff = None, 3
        for attempt in range(self.max_retries):
            try:
                text = self._raw_call(deployment_name, messages, json_mode)
                text = THINK_TAG.sub("", text).strip()
                if not text:
                    raise ValueError("empty completion")
                try:
                    return json.loads(text)
                except Exception:
                    got = self._extract_json(text)
                    if got:
                        return {"features": got} if isinstance(got, list) else got
                    raise ValueError(f"unparseable: {text[:200]}")

            except BadRequestError as e:
                msg = str(e)
                if self._think_supported and "think" in msg:
                    self._think_supported = False
                    continue
                if json_mode and "json_object" in msg:
                    json_mode = False
                    continue
                raise

            except Exception as e:
                last = e
                if attempt < self.max_retries - 1:
                    time.sleep(backoff); backoff *= 2
                    continue
        raise RuntimeError(f"failed after {self.max_retries} attempts: {last}")


provider = StreamingLocalProvider(
    base_url=BASE_URL, api_key=API_KEY, default_text_model=MODEL_B,
    temperature=TEMPERATURE, max_tokens=MAX_TOKENS, max_retries=4, stream=True,
)
print("provider ready:", type(provider).__name__, "| model:", provider.text_model)

provider ready: StreamingLocalProvider | model: qwen3.6-35b


## Block 4 — calibration on MODEL_B (timing + groundedness gate)

In [5]:
EVIDENCE_FIELDS = ["named_entities", "specific_action_verbs", "generic_verbs",
                   "locative_expressions", "hedge_spans", "deictic_spans",
                   "metacomment_spans", "diminutive_or_affective_forms",
                   "quantity_expressions"]
REQUIRED = EVIDENCE_FIELDS + ["complete_propositions", "regions_referenced",
                              "repeated_content_lemmas", "self_corrections"]

def extract_one(text: str) -> dict:
    return provider.text_features([text], prompt=V2_PROMPT)[0]

def groundedness(obj: dict, text: str):
    low, hit, tot = text.lower(), 0, 0
    for f in EVIDENCE_FIELDS:
        for s in obj.get(f, []) or []:
            if isinstance(s, str) and s.strip():
                tot += 1
                hit += s.strip().lower() in low
    return hit / tot if tot else np.nan

def missing_fields(obj: dict):
    return [f for f in REQUIRED if f not in obj]

if RUN_LLM:
    calib = docs[docs.label.isna()].head(N_CALIB)
    probe = []
    for _, r in calib.iterrows():
        s = time.time()
        try:
            o = extract_one(r.text)
            probe.append({"file": r.file, "sec": time.time() - s, "ok": True,
                          "missing": missing_fields(o), "grounded": groundedness(o, r.text)})
        except Exception as e:
            probe.append({"file": r.file, "sec": time.time() - s, "ok": False,
                          "missing": None, "grounded": np.nan,
                          "err": f"{type(e).__name__}: {e}"[:120]})
    probe = pd.DataFrame(probe)
    print(probe.to_string(index=False))
    med = probe.sec.median()
    print(f"\nmedian {med:.1f}s/doc  ->  327 docs ~ {med*327/60:.0f} min")
    print(f"parsed OK: {probe.ok.sum()}/{len(probe)}   mean groundedness: {probe.grounded.mean():.2f}")
    print("\n>>> GATE: groundedness should stay > 0.80, same as run A.")
else:
    print("RUN_LLM = False — skipping calibration.")

RUN_LLM = False — skipping calibration.


## Block 5 — full MODEL_B extraction (resumable)

Same resume logic as notebook 08: one JSON line per doc, flushed immediately, already-done
files skipped on re-run.

In [6]:
def load_raw(path: Path) -> dict:
    out = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                try:
                    rec = json.loads(line); out[rec["file"]] = rec
                except Exception:
                    pass
    return out

def seal_last_line(path: Path):
    if path.exists() and path.stat().st_size:
        with path.open("rb+") as fh:
            fh.seek(-1, 2)
            if fh.read(1) != b"\n":
                fh.write(b"\n")

if RUN_LLM:
    seal_last_line(RAW_B)
    done = load_raw(RAW_B)
    todo = docs[~docs.file.isin(done)]
    print(f"already done: {len(done)}   remaining: {len(todo)}")
    t0 = time.time()
    with RAW_B.open("a", encoding="utf-8") as fh:
        for i, (_, r) in enumerate(todo.iterrows(), 1):
            rec = {"file": r.file, "label": (None if pd.isna(r.label) else int(r.label)),
                   "model": MODEL_B, "prompt_sha": PROMPT_SHA}
            try:
                rec["response"] = extract_one(r.text); rec["error"] = None
            except Exception as e:
                rec["response"] = None; rec["error"] = f"{type(e).__name__}: {e}"[:300]
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()
            if i % 10 == 0 or i == len(todo):
                el = time.time() - t0
                print(f"  {i}/{len(todo)}  {el/60:.1f} min, ~{el/i*(len(todo)-i)/60:.1f} min left",
                      flush=True)

rawB = load_raw(RAW_B)
nB_ok = sum(v["response"] is not None for v in rawB.values())
print(f"\nMODEL_B extracted: {nB_ok}/{len(rawB)} ok, {len(rawB) - nB_ok} failed")
json.dump({"model": MODEL_B, "base_url": BASE_URL, "temperature": TEMPERATURE,
           "max_tokens": MAX_TOKENS, "prompt_sha256_16": PROMPT_SHA,
           "n_documents": len(rawB), "n_succeeded": nB_ok, "n_failed": len(rawB) - nB_ok,
           "test_set_used": False,
           "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")},
          META_B.open("w"), indent=2)
print("metadata ->", META_B)


MODEL_B extracted: 327/327 ok, 0 failed
metadata -> fileDataset/outputs/v2b_run_metadata.json


## Block 6 — build both feature matrices

Identical `to_row` mapping as notebook 08. `dfA` = run A (`qwen3.5:122b`), `dfB` = run B
(`qwen3.6-35b`). Counts → per-100-word rates + two scale-free ratios.

In [7]:
LIST_FIELDS = {
    "n_entities":     "named_entities",
    "n_specific_verb":"specific_action_verbs",
    "n_generic_verb": "generic_verbs",
    "n_locative":     "locative_expressions",
    "n_hedge":        "hedge_spans",
    "n_deictic":      "deictic_spans",
    "n_metacomment":  "metacomment_spans",
    "n_diminutive":   "diminutive_or_affective_forms",
    "n_quantity":     "quantity_expressions",
}
INT_FIELDS = {"n_proposition": "complete_propositions", "n_selfcorrect": "self_corrections"}

def to_row(obj: dict) -> dict:
    r = {}
    for out, key in LIST_FIELDS.items():
        v = obj.get(key) or []
        r[out] = len(v) if isinstance(v, list) else 0
    for out, key in INT_FIELDS.items():
        v = obj.get(key, 0)
        r[out] = int(v) if isinstance(v, (int, float)) else 0
    reg = obj.get("regions_referenced") or []
    r["n_region"] = len(set(reg)) if isinstance(reg, list) else 0
    rep = obj.get("repeated_content_lemmas") or []
    r["n_repeated_lemma"] = sum(1 for x in rep if isinstance(x, dict) and x.get("lemma"))
    r["repeat_mass"] = sum(int(x.get("count", 0)) for x in rep
                           if isinstance(x, dict) and str(x.get("count", "")).isdigit())
    return r

def build_df(path: Path) -> pd.DataFrame:
    raw = load_raw(path)
    ok  = {f: v["response"] for f, v in raw.items() if v["response"] is not None}
    feat = pd.DataFrame([{"file": f, **to_row(o)} for f, o in ok.items()])
    d = docs.merge(feat, on="file", how="inner")
    for c in COUNTS:
        d[c + "_r100"] = 100 * d[c] / d["n_word"].clip(lower=1)
    d["specific_verb_ratio"] = d.n_specific_verb / (d.n_specific_verb + d.n_generic_verb).clip(lower=1)
    d["region_breadth"] = d.n_region / 3.0
    return d

COUNTS = list(LIST_FIELDS) + list(INT_FIELDS) + ["n_region", "n_repeated_lemma", "repeat_mass"]
RATES  = [c + "_r100" for c in COUNTS]
RATIOS = ["specific_verb_ratio", "region_breadth"]

dfA = build_df(RAW_A)
dfB = build_df(RAW_B)
print(f"run A (MODEL_A): {len(dfA)} docs   run B (MODEL_B): {len(dfB)} docs")
print(f"features: {len(COUNTS)} counts + {len(RATES)} rates + {len(RATIOS)} ratios")

# labelled views, aligned to the SAME file order
labA = dfA[dfA.label.notna()].copy(); labA["label"] = labA.label.astype(int)
labB = dfB[dfB.label.notna()].copy(); labB["label"] = labB.label.astype(int)
common_lab = sorted(set(labA.file) & set(labB.file))
labA = labA.set_index("file").loc[common_lab].reset_index()
labB = labB.set_index("file").loc[common_lab].reset_index()
y = labA.label.values
print(f"labelled docs present in BOTH runs: {len(common_lab)}  (pos {y.sum()}, neg {(y==0).sum()})")

run A (MODEL_A): 327 docs   run B (MODEL_B): 327 docs
features: 14 counts + 14 rates + 2 ratios
labelled docs present in BOTH runs: 241  (pos 70, neg 171)


## Block 7 — feature stability (run A vs run B)

Per feature, on every document present in both runs:

* **Spearman ρ** — rank agreement (the headline number).
* **ICC(2,1)** — two-way random effects, absolute agreement, single rating: penalises
  systematic offsets between models, not just rank disagreement.
* **exact-match** — share of docs where the two integer counts are identical.
* **median A / median B** — to see systematic level shifts.

In [8]:
from scipy import stats

A_all = dfA.set_index("file"); B_all = dfB.set_index("file")
common = sorted(set(A_all.index) & set(B_all.index))
A = A_all.loc[common]; B = B_all.loc[common]
print(f"documents compared: {len(common)}")

def icc21(a, b):
    Y = np.column_stack([a, b]).astype(float)
    n, k = Y.shape
    gm = Y.mean()
    SSR = k * ((Y.mean(1) - gm) ** 2).sum()          # between targets (docs)
    SSC = n * ((Y.mean(0) - gm) ** 2).sum()          # between raters (models)
    SSE = ((Y - gm) ** 2).sum() - SSR - SSC
    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))
    denom = MSR + (k - 1) * MSE + k * (MSC - MSE) / n
    return np.nan if denom == 0 else (MSR - MSE) / denom

rows = []
for f in COUNTS + RATES + RATIOS:
    a, b = A[f].astype(float).values, B[f].astype(float).values
    if a.std() == 0 and b.std() == 0:
        rho = pear = np.nan
    else:
        rho  = stats.spearmanr(a, b).correlation
        pear = np.corrcoef(a, b)[0, 1] if (a.std() and b.std()) else np.nan
    exact = np.mean(np.isclose(a, b)) if f in COUNTS else np.nan
    rows.append({"feature": f, "spearman": rho, "pearson": pear, "icc21": icc21(a, b),
                 "exact_match": exact, "medA": np.median(a), "medB": np.median(b),
                 "mean_abs_diff": np.mean(np.abs(a - b))})
stab = pd.DataFrame(rows).set_index("feature")
stab["stable"] = stab.spearman >= STABILITY_MIN_RHO

pd.set_option("display.width", 200, "display.max_rows", 80)
print(stab.round(3).sort_values("spearman", ascending=False).to_string())

n_stable = int(stab.loc[COUNTS + RATIOS, "stable"].sum())
print(f"\ncount+ratio features stable (rho >= {STABILITY_MIN_RHO}): "
      f"{n_stable}/{len(COUNTS) + len(RATIOS)}")
print("median Spearman over count features:",
      round(stab.loc[COUNTS, 'spearman'].median(), 3))
print("UNSTABLE count/ratio features:",
      list(stab.loc[COUNTS + RATIOS].query("not stable").index))
stab.to_csv(OUT / "v2_stability.csv")

documents compared: 327
                       spearman  pearson  icc21  exact_match    medA    medB  mean_abs_diff  stable
feature                                                                                            
n_proposition             0.900    0.878  0.866        0.459  13.000  12.000          1.300    True
n_locative                0.881    0.888  0.882        0.367   7.000   7.000          1.205    True
n_quantity                0.878    0.839  0.839        0.777   0.000   0.000          0.370    True
n_quantity_r100           0.870    0.829  0.828          NaN   0.000   0.000          0.491    True
n_hedge                   0.811    0.814  0.808        0.624   1.000   1.000          0.584    True
n_proposition_r100        0.802    0.817  0.804          NaN  18.421  17.391          1.892    True
n_locative_r100           0.797    0.823  0.816          NaN   9.524  10.145          1.618    True
n_metacomment             0.791    0.673  0.669        0.758   1.000   0.000

## Block 8 — automatic degeneracy gate  (`zero_share > 0.70`)

Applied to the **labelled** docs, in **either** run. Dropped count features also lose their
`_r100` rate. This is the only automatic drop; the stability flag from Block 7 is carried as
metadata and used for a sensitivity arm, not for dropping.

In [9]:
zsA = (labA[COUNTS] == 0).mean()
zsB = (labB[COUNTS] == 0).mean()
gate = pd.DataFrame({"zero_share_A": zsA, "zero_share_B": zsB})
gate["degenerate"] = (gate.zero_share_A > ZERO_SHARE_MAX) | (gate.zero_share_B > ZERO_SHARE_MAX)
gate["stable"] = stab.loc[COUNTS, "stable"]
print(gate.round(3).sort_values("zero_share_A", ascending=False).to_string())

DROP_COUNTS = gate.index[gate.degenerate].tolist()
KEPT_COUNTS = [c for c in COUNTS if c not in DROP_COUNTS]
KEPT_RATES  = [c + "_r100" for c in KEPT_COUNTS]
KEPT_RATIOS = list(RATIOS)                      # ratios are scale-free, keep both
print(f"\ndropped by degeneracy gate : {DROP_COUNTS}")
print(f"kept counts ({len(KEPT_COUNTS)})     : {KEPT_COUNTS}")

# frozen V2 representation = length-normalised, non-degenerate
V2_RATES   = KEPT_RATES + KEPT_RATIOS                                  # primary
V2_ALL     = KEPT_COUNTS + KEPT_RATES + KEPT_RATIOS                    # counts+rates (secondary)
V2_STABLE  = ([c + "_r100" for c in KEPT_COUNTS if stab.loc[c, "stable"]]
              + [r for r in KEPT_RATIOS if stab.loc[r, "stable"]])     # stable-only sensitivity
print(f"\nV2_RATES  ({len(V2_RATES)}): {V2_RATES}")
print(f"V2_STABLE ({len(V2_STABLE)}): {V2_STABLE}")

                  zero_share_A  zero_share_B  degenerate  stable
n_entities               0.946         0.386        True   False
n_selfcorrect            0.622         0.556       False    True
n_quantity               0.544         0.519       False    True
n_metacomment            0.510         0.535       False    True
n_hedge                  0.344         0.423       False    True
n_deictic                0.261         0.423       False    True
n_generic_verb           0.183         0.154       False   False
n_region                 0.087         0.108       False    True
n_diminutive             0.083         0.108       False    True
n_proposition            0.029         0.029       False    True
n_locative               0.025         0.021       False    True
n_specific_verb          0.017         0.012       False    True
n_repeated_lemma         0.012         0.004       False    True
repeat_mass              0.012         0.004       False    True

dropped by degeneracy ga

## Block 9 — classifier comparison

Same CV protocol as notebooks 04–08: **10× repeated stratified 5-fold**, out-of-fold
probabilities averaged over repeats, AUC as the primary metric with a bootstrap 95% CI, plus
**paired** bootstrap Δ between the representations that matter for the thesis question.

* **Primary classifier — Elastic-Net logistic regression** (`LogisticRegressionCV`,
  `penalty="elasticnet"`, `solver="saga"`, `l1_ratio` and `C` tuned by inner CV inside every
  outer fold — no leakage).
* **Robustness — plain L2 logistic regression and Linear SVM** (SVM calibrated to
  probabilities via inner CV) on the representations of interest.
* All vectorisers / encoders sit **inside** the pipeline, so nothing is fitted on the whole
  set before CV.

Features come from **run A** (the reference run the frozen pipeline will consume).

In [10]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score

N_REPEATS, N_SPLITS = 10, 5

def enet():
    return make_pipeline(StandardScaler(), LogisticRegressionCV(
        penalty="elasticnet", solver="saga", l1_ratios=[0.3, 0.6, 0.9], Cs=8,
        max_iter=5000, class_weight="balanced", scoring="roc_auc",
        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1))

def l2():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced"))

def lsvm():
    return make_pipeline(StandardScaler(), CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", max_iter=5000), method="sigmoid", cv=3))

def cv_proba(model, X):
    acc = np.zeros(len(y))
    for r in range(N_REPEATS):
        cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED + r)
        acc += cross_val_predict(model, X, y, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
    return acc / N_REPEATS

def boot_ci(p, n=2000):
    rng = np.random.default_rng(SEED); a = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            a.append(roc_auc_score(y[i], p[i]))
    return np.percentile(a, [2.5, 97.5])

def evaluate(name, clf_name, model, X):
    p = cv_proba(model, X)
    lo, hi = boot_ci(p)
    return {"representation": name, "clf": clf_name,
            "n_feat": (X.shape[1] if getattr(X, "ndim", 1) == 2 else "text"),
            "AUC": roc_auc_score(y, p), "CI_low": lo, "CI_high": hi,
            "macroF1": f1_score(y, p > 0.5, average="macro"),
            "balAcc": balanced_accuracy_score(y, p > 0.5)}, p

PROBA = {}
res = []
def run(name, clf_name, model, X, key):
    row, p = evaluate(name, clf_name, model, X)
    PROBA[key] = p; res.append(row)
    print(f"  {name:22s} [{clf_name:5s}]  AUC {row['AUC']:.3f}  CI {row['CI_low']:.3f}-{row['CI_high']:.3f}")

X_len  = labA[["n_word"]].values
X_surf = labA[SURFACE].values

# ---- primary: Elastic Net across all representations ----
print("Elastic Net (primary):")
run("Transcript length", "enet", enet(), X_len,                      "len")
run("Surface stats",     "enet", enet(), X_surf,                     "surf")
run("V2 rates",          "enet", enet(), labA[V2_RATES].values,      "v2r")
run("V2 rates + surface","enet", enet(), labA[V2_RATES + SURFACE].values, "v2r_surf")
run("V2 counts+rates",   "enet", enet(), labA[V2_ALL].values,        "v2all")
run("V2 stable-only",    "enet", enet(), labA[V2_STABLE].values,     "v2stab")

# V1 categorical (leak-free OHE pipeline)
v1 = pd.read_csv(next(p for p in [Path("../OutputsQwen/train_all_feature_values.csv"),
                                  Path("OutputsQwen/train_all_feature_values.csv")] if p.exists())
                 ).rename(columns={"File": "file"})
v1c = [c for c in v1.columns if c not in ("file", "Class", "raw_llm_output")]
m1 = labA[["file"]].merge(v1[["file"] + v1c], on="file", how="left").fillna("missing")
run("V1 categorical", "enet",
    make_pipeline(OneHotEncoder(handle_unknown="ignore"),
                  LogisticRegression(max_iter=2000, class_weight="balanced")),
    m1[v1c], "v1")

# TF-IDF ceiling (leak-free: vectoriser inside the pipeline)
run("TF-IDF char 3-5gram", "l2",
    make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True),
                  LogisticRegression(max_iter=3000, class_weight="balanced")),
    labA.text.values, "tfidf")

# ---- robustness: L2 and Linear SVM on the representations that decide the question ----
print("\nRobustness (L2 logreg, Linear SVM):")
for cname, ctor in [("l2", l2), ("svm", lsvm)]:
    run("Surface stats",      cname, ctor(), X_surf,                        f"surf_{cname}")
    run("V2 rates",           cname, ctor(), labA[V2_RATES].values,         f"v2r_{cname}")
    run("V2 rates + surface", cname, ctor(), labA[V2_RATES + SURFACE].values, f"v2r_surf_{cname}")

table = pd.DataFrame(res).sort_values("AUC", ascending=False).reset_index(drop=True)
print()
print(table.round(3).to_string(index=False))
table.to_csv(OUT / "v2_stability_comparison.csv", index=False)

Elastic Net (primary):


  Transcript length      [enet ]  AUC 0.691  CI 0.614-0.764


  Surface stats          [enet ]  AUC 0.743  CI 0.673-0.803


  V2 rates               [enet ]  AUC 0.768  CI 0.691-0.840


  V2 rates + surface     [enet ]  AUC 0.805  CI 0.737-0.868


...

  V2 rates               [l2   ]  AUC 0.768  CI 0.690-0.837


  V2 rates + surface     [l2   ]  AUC 0.786  CI 0.716-0.851


  Surface stats          [svm  ]  AUC 0.749  CI 0.681-0.809


  V2 rates               [svm  ]  AUC 0.765  CI 0.686-0.836


  V2 rates + surface     [svm  ]  AUC 0.778  CI 0.706-0.845

     representation  clf n_feat   AUC  CI_low  CI_high  macroF1  balAcc
TF-IDF char 3-5gram   l2   text 0.867   0.816    0.912    0.758   0.753
 V2 rates + surface enet     22 0.805   0.737    0.868    0.744   0.756
    V2 counts+rates enet     28 0.795   0.718    0.862    0.766   0.770
 V2 rates + surface   l2     22 0.786   0.716    0.851    0.719   0.729
 V2 rates + surface  svm     22 0.778   0.706    0.845    0.669   0.652
     V2 stable-only enet     13 0.776   0.699    0.847    0.737   0.745
           V2 rates enet     15 0.768   0.691    0.840    0.737   0.745
           V2 rates   l2     15 0.768   0.690    0.837    0.729   0.739
           V2 rates  svm     15 0.765   0.686    0.836    0.649   0.636
      Surface stats  svm      7 0.749   0.681    0.809    0.606   0.602
      Surface stats enet      7 0.743   0.673    0.803    0.642   0.662
      Surface stats   l2      7 0.742   0.673    0.803    0.644   0.672
   

### Paired bootstrap — the decisive contrasts

In [11]:
def paired(a, b, n=5000):
    rng = np.random.default_rng(SEED); d = []
    pa, pb = PROBA[a], PROBA[b]
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        d.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    d = np.array(d)
    return d.mean(), np.percentile(d, [2.5, 97.5]), (d <= 0).mean()

CONTRASTS = [
    ("v2r_surf", "surf",  "V2 rates + surface  vs  surface alone   (does V2 add to surface?)"),
    ("v2r",      "surf",  "V2 rates            vs  surface alone"),
    ("v2r",      "len",   "V2 rates            vs  transcript length"),
    ("v2r",      "v1",    "V2 rates            vs  V1 categorical"),
    ("tfidf",    "v2r",   "TF-IDF              vs  V2 rates"),
    ("v2stab",   "surf",  "V2 stable-only      vs  surface alone"),
    ("v2r_surf_l2",  "surf_l2",  "[L2]  V2 rates+surface vs surface"),
    ("v2r_surf_svm", "surf_svm", "[SVM] V2 rates+surface vs surface"),
]
print(f"{'contrast':58s} {'dAUC':>8s}  {'95% CI':>18s}  {'P(d<=0)':>8s}")
for a, b, label in CONTRASTS:
    if a in PROBA and b in PROBA:
        m, ci, pneg = paired(a, b)
        flag = "  *" if (ci[0] > 0 or ci[1] < 0) else ""
        print(f"{label:58s} {m:+.4f}  [{ci[0]:+.4f},{ci[1]:+.4f}]  {pneg:6.3f}{flag}")

contrast                                                       dAUC              95% CI   P(d<=0)


V2 rates + surface  vs  surface alone   (does V2 add to surface?) +0.0627  [+0.0042,+0.1195]   0.019  *


V2 rates            vs  surface alone                      +0.0252  [-0.0417,+0.0896]   0.224


V2 rates            vs  transcript length                  +0.0771  [-0.0086,+0.1645]   0.038


V2 rates            vs  V1 categorical                     +0.0750  [-0.0060,+0.1565]   0.036


TF-IDF              vs  V2 rates                           +0.0997  [+0.0364,+0.1655]   0.001  *


V2 stable-only      vs  surface alone                      +0.0328  [-0.0343,+0.0970]   0.159


[L2]  V2 rates+surface vs surface                          +0.0443  [-0.0191,+0.1073]   0.088


[SVM] V2 rates+surface vs surface                          +0.0293  [-0.0357,+0.0915]   0.174


## Block 9b — corrected frozen-primary feature set (stability-gated)

Block 10 froze `frozen_primary = V2_RATES + SURFACE`, which still carries two features Block 7/8 flagged as model-dependent: `specific_verb_ratio` (Spearman ρ = 0.44, BH-significant only under 122b) and `n_generic_verb_r100` (ρ = 0.34; gated out via its count `n_generic_verb`, ρ = 0.42). The frozen primary is corrected here to `V2_STABLE + SURFACE` **before** any test-set code. Same Block 9 protocol — 10×5 repeated CV, identical folds/seed, leak-free pipelines. The 61-case test set is not read.

In [12]:
# --- step 1: FROZEN_PRIMARY_FEATURES, derived from the gate's own output on disk
# Re-derive V2_STABLE from v2_stability.csv using Block 8's exact rule (stability flag
# read on the COUNT, its _r100 rate carried along) so the list tracks the gate output
# rather than a pinned literal that could drift.
stab_disk = pd.read_csv(OUT / "v2_stability.csv", index_col="feature")
V2_STABLE_DISK = ([c + "_r100" for c in KEPT_COUNTS if bool(stab_disk.loc[c, "stable"])]
                  + [r for r in KEPT_RATIOS if bool(stab_disk.loc[r, "stable"])])
assert V2_STABLE_DISK == V2_STABLE, (
    f"gate output on disk {V2_STABLE_DISK} != in-memory V2_STABLE {V2_STABLE}")

FROZEN_PRIMARY_FEATURES = V2_STABLE + SURFACE
assert "specific_verb_ratio" not in FROZEN_PRIMARY_FEATURES
assert "n_generic_verb_r100" not in FROZEN_PRIMARY_FEATURES
assert len(FROZEN_PRIMARY_FEATURES) == len(V2_STABLE) + len(SURFACE) == 20
_removed = [f for f in (V2_RATES + SURFACE) if f not in FROZEN_PRIMARY_FEATURES]
assert _removed == ["n_generic_verb_r100", "specific_verb_ratio"], _removed
print(f"FROZEN_PRIMARY_FEATURES ({len(FROZEN_PRIMARY_FEATURES)} cols):")
print(" ", FROZEN_PRIMARY_FEATURES)
print(f"dropped vs old V2_RATES+SURFACE primary: {_removed}")

# --- step 3: re-run the 10x5 CV on the corrected set (Block 9 protocol) ---------
print("\nElastic Net, 10x5 repeated CV (same folds/seed as Block 9):")
row_fp, p_fp = evaluate("V2 stable + surface (frozen primary)", "enet",
                        enet(), labA[FROZEN_PRIMARY_FEATURES].values)
PROBA["v2stab_surf"] = p_fp
if not any(r["representation"] == row_fp["representation"] for r in res):
    res.append(row_fp)
print(f"  V2 stable + surface   AUC {row_fp['AUC']:.3f}  "
      f"CI {row_fp['CI_low']:.3f}-{row_fp['CI_high']:.3f}  (n_feat {row_fp['n_feat']})")

# reproducibility gate: the published number for this arm is the "V2 stable-only" row
# (V2_STABLE, 13 feats) in v2_stability_comparison.csv -> 0.776. Re-derive it on a fresh
# CV run (PROBA['v2stab'] from Block 9, same folds) and require it to land inside its
# reported bootstrap CI band; require the frozen primary (a superset) to land there too.
# Stop rather than freeze if either fails.
cmp_disk = pd.read_csv(OUT / "v2_stability_comparison.csv")
_ref = cmp_disk[(cmp_disk.representation == "V2 stable-only") &
                (cmp_disk.clf == "enet")].iloc[0]
_auc_stab_only = roc_auc_score(y, PROBA["v2stab"])
print(f"\n  reproduce 'V2 stable-only'   : fresh AUC {_auc_stab_only:.4f}  vs reported "
      f"{_ref.AUC:.4f}   band [{_ref.CI_low:.3f}, {_ref.CI_high:.3f}]")
assert _ref.CI_low <= _auc_stab_only <= _ref.CI_high, (
    f"stable-only CV AUC {_auc_stab_only:.4f} outside reported CI band "
    f"[{_ref.CI_low:.3f}, {_ref.CI_high:.3f}] -- STOP, do not freeze")
assert _ref.CI_low <= row_fp["AUC"] <= _ref.CI_high, (
    f"frozen-primary CV AUC {row_fp['AUC']:.4f} outside stable-only CI band "
    f"[{_ref.CI_low:.3f}, {_ref.CI_high:.3f}] -- STOP, do not freeze")
print(f"  frozen primary (stable+surf) : AUC {row_fp['AUC']:.4f}  -> in band")
print("  reproducibility + band check : OK")

# refreshed comparison table (adds the frozen-primary row; feeds Block 10's spec)
table = pd.DataFrame(res).sort_values("AUC", ascending=False).reset_index(drop=True)
table.to_csv(OUT / "v2_stability_comparison.csv", index=False)

# --- step 4: the decisive paired bootstrap missing from Block 9 ----------------
# V2_STABLE + surface  vs  surface alone, ElasticNet, 2000 resamples, on the SAME
# per-document OOF prediction pairs used for every other Block 9 paired contrast
# (paired() indexes PROBA[...] and shares SEED across contrasts).
_m, _ci, _pneg = paired("v2stab_surf", "surf", n=2000)
PAIRED_FROZEN_VS_SURFACE = {
    "contrast": "V2_STABLE + surface (enet)  vs  surface alone (enet)",
    "n_resamples": 2000, "delta_auc": float(_m),
    "ci95": [float(_ci[0]), float(_ci[1])], "p_delta_le_0": float(_pneg),
}
_mo, _cio, _pnego = paired("v2r_surf", "surf", n=2000)     # deprecated primary, for context
PAIRED_OLD_PRIMARY_VS_SURFACE = {
    "contrast": "V2_RATES + surface (enet)  vs  surface alone (enet)  [deprecated primary]",
    "n_resamples": 2000, "delta_auc": float(_mo),
    "ci95": [float(_cio[0]), float(_cio[1])], "p_delta_le_0": float(_pnego),
}
print("\npaired bootstrap (2000 resamples, same pairs as Block 9):")
print(f"  V2_STABLE+surface vs surface : dAUC {_m:+.4f}  95% CI [{_ci[0]:+.4f}, {_ci[1]:+.4f}]"
      f"  P(dAUC<=0) = {_pneg:.3f}{'   *' if _ci[0] > 0 else ''}")
print(f"  (context) old primary vs surf: dAUC {_mo:+.4f}  95% CI [{_cio[0]:+.4f}, {_cio[1]:+.4f}]"
      f"  P(dAUC<=0) = {_pnego:.3f}")
print("\n>>> decisive number for 'does V2 add beyond surface stats':")
print(f"    dAUC = {_m:+.4f}   95% CI [{_ci[0]:+.4f}, {_ci[1]:+.4f}]   P(dAUC<=0) = {_pneg:.3f}")


FROZEN_PRIMARY_FEATURES (20 cols):
  ['n_specific_verb_r100', 'n_locative_r100', 'n_hedge_r100', 'n_deictic_r100', 'n_metacomment_r100', 'n_diminutive_r100', 'n_quantity_r100', 'n_proposition_r100', 'n_selfcorrect_r100', 'n_region_r100', 'n_repeated_lemma_r100', 'repeat_mass_r100', 'region_breadth', 'n_word', 'n_type', 'ttr', 'n_sent', 'mlu', 'n_comma', 'n_char']
dropped vs old V2_RATES+SURFACE primary: ['n_generic_verb_r100', 'specific_verb_ratio']

Elastic Net, 10x5 repeated CV (same folds/seed as Block 9):


  V2 stable + surface   AUC 0.806  CI 0.739-0.867  (n_feat 20)

  reproduce 'V2 stable-only'   : fresh AUC 0.7755  vs reported 0.7755   band [0.699, 0.847]
  frozen primary (stable+surf) : AUC 0.8058  -> in band
  reproducibility + band check : OK



paired bootstrap (2000 resamples, same pairs as Block 9):
  V2_STABLE+surface vs surface : dAUC +0.0635  95% CI [+0.0065, +0.1197]  P(dAUC<=0) = 0.016   *
  (context) old primary vs surf: dAUC +0.0628  95% CI [+0.0059, +0.1183]  P(dAUC<=0) = 0.017

>>> decisive number for 'does V2 add beyond surface stats':
    dAUC = +0.0635   95% CI [+0.0065, +0.1197]   P(dAUC<=0) = 0.016


## Block 10 — freeze the pipeline

Everything needed to reproduce the result and to run the **single** future test-set
evaluation is written to `fileDataset/outputs/frozen_pipeline_v2/`:

* `frozen_spec.json` — models, prompt SHA, gate thresholds, feature lists, CV protocol,
  classifier params, stability + comparison summaries.
* `pipe_<name>.joblib` — the fitted pipelines (surface baseline, V2 rates + surface, TF-IDF
  ceiling), trained on all labelled run-A docs.

The 61 test transcripts are **not** read here.

In [13]:
import joblib, sklearn

# tf-idf ceiling pipeline (unchanged)
_tfidf_pipe = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=3000, class_weight="balanced"))

# frozen primary is now the stability-gated set from Block 9b, not V2_RATES + SURFACE
FROZEN = {
    "surface":           (l2(),   SURFACE,                 labA[SURFACE].values),
    "v2_stable_surface": (enet(), FROZEN_PRIMARY_FEATURES, labA[FROZEN_PRIMARY_FEATURES].values),
    "tfidf_char35":      (_tfidf_pipe, "text",             labA.text.values),
}
for name, (model, feats, X) in FROZEN.items():
    model.fit(X, y)
    joblib.dump({"pipeline": model, "features": feats, "feature_source_run": "A / " + MODEL_A},
                FROZEN_DIR / f"pipe_{name}.joblib")
    print("froze", name)

# --- deprecate (never delete) the superseded V2_RATES + SURFACE primary --------
_old = FROZEN_DIR / "pipe_v2_rates_surface.joblib"
_dep = FROZEN_DIR / "pipe_v2_rates_surface_deprecated.joblib"
_dep_reason = ("superseded by pipe_v2_stable_surface.joblib: carried features flagged unstable "
               "in Block 7/8 (specific_verb_ratio rho=0.44, n_generic_verb_r100 rho=0.34)")
if _old.exists():
    _old.replace(_dep)
    print("deprecated existing artifact ->", _dep.name)
else:
    _m_dep = enet(); _m_dep.fit(labA[V2_RATES + SURFACE].values, y)
    joblib.dump({"pipeline": _m_dep, "features": V2_RATES + SURFACE,
                 "feature_source_run": "A / " + MODEL_A,
                 "deprecated": True, "reason": _dep_reason}, _dep)
    print("wrote audit copy ->", _dep.name)

MODEL_ARTIFACT = "pipe_v2_stable_surface.joblib"

spec = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "prompt_sha256_16": PROMPT_SHA,
    "prompt_sha256_assert": EXPECTED_PROMPT_SHA,          # still f52aae5d939d620e (asserted in Block 1)
    "models": {"A": MODEL_A, "B": MODEL_B},
    "test_extraction_model": MODEL_A,                     # qwen3.5:122b -- the run the frozen pipe was trained on
                                                         # (MODEL_B qwen3.6-35b was only the stability comparator)
    "model_artifact": MODEL_ARTIFACT,
    "deprecated_artifacts": {_dep.name: _dep_reason},
    "llm_params": {"temperature": TEMPERATURE, "max_tokens": MAX_TOKENS, "stream": True,
                   "think": False},
    "gates": {"zero_share_max": ZERO_SHARE_MAX, "stability_min_rho": STABILITY_MIN_RHO},
    "dropped_degenerate": DROP_COUNTS,                                        # ['n_entities']  (Block 8, verbatim)
    "unstable_features": list(stab.loc[COUNTS + RATIOS].query("not stable").index),  # Block 7, verbatim
    "feature_sets": {"V2_RATES": V2_RATES, "V2_STABLE": V2_STABLE, "SURFACE": SURFACE,
                     "frozen_primary": FROZEN_PRIMARY_FEATURES},             # corrected: V2_STABLE + SURFACE
    "cv_protocol": {"scheme": "repeated stratified k-fold", "repeats": N_REPEATS,
                    "folds": N_SPLITS, "seed": SEED, "metric": "roc_auc",
                    "ci": "2000x subject bootstrap"},
    "classifiers": {
        "primary": "LogisticRegressionCV(penalty=elasticnet, solver=saga, "
                   "l1_ratios=[.3,.6,.9], Cs=8, class_weight=balanced), StandardScaler",
        "robustness": ["LogisticRegression(l2, class_weight=balanced)",
                       "CalibratedClassifierCV(LinearSVC(class_weight=balanced))"]},
    "stability_summary": {
        "docs_compared": len(common),
        "median_spearman_counts": float(stab.loc[COUNTS, "spearman"].median()),
        "n_stable_count_ratio": int(stab.loc[COUNTS + RATIOS, "stable"].sum()),
        "n_total_count_ratio": len(COUNTS) + len(RATIOS)},
    "frozen_primary_cv_auc": float(row_fp["AUC"]),
    "paired_bootstrap": {
        "frozen_primary_vs_surface": PAIRED_FROZEN_VS_SURFACE,
        "deprecated_primary_vs_surface": PAIRED_OLD_PRIMARY_VS_SURFACE},
    "comparison_auc": table.set_index(["representation", "clf"]).AUC.round(4).to_dict().__repr__(),
    "test_set_used": False,
}
json.dump(spec, (FROZEN_DIR / "frozen_spec.json").open("w"), indent=2, default=str)
print("\nfrozen_spec.json ->", FROZEN_DIR / "frozen_spec.json")
print("frozen_primary :", len(spec["feature_sets"]["frozen_primary"]), "cols  ->", MODEL_ARTIFACT)
_pb = PAIRED_FROZEN_VS_SURFACE
print(f"paired dAUC    : {_pb['delta_auc']:+.4f}  CI {_pb['ci95']}  P(<=0) {_pb['p_delta_le_0']:.3f}")


froze surface


froze v2_stable_surface
froze tfidf_char35
deprecated existing artifact -> pipe_v2_rates_surface_deprecated.joblib

frozen_spec.json -> fileDataset/outputs/frozen_pipeline_v2/frozen_spec.json
frozen_primary : 20 cols  -> pipe_v2_stable_surface.joblib
paired dAUC    : +0.0635  CI [0.006543441539207834, 0.11973858587914607]  P(<=0) 0.016


## Block 10b — final consistency check

Reload `frozen_spec.json` and `pipe_v2_stable_surface.joblib` from disk and assert they agree — same reload-and-assert pattern as Block 0–1 (`assert RAW_A.exists()`, `assert PROMPT_SHA == EXPECTED_PROMPT_SHA`). No test-set code.

In [14]:
_spec = json.load((FROZEN_DIR / "frozen_spec.json").open())
_art  = joblib.load(FROZEN_DIR / _spec["model_artifact"])
_fp   = _spec["feature_sets"]["frozen_primary"]
_pipe = _art["pipeline"]

# 1) artifact <-> spec feature-list agreement, exact order + pipeline arity
assert _art["features"] == _fp, "artifact 'features' != spec frozen_primary"
assert list(_fp) == list(V2_STABLE) + list(SURFACE), "frozen_primary != V2_STABLE + SURFACE"
assert _pipe.n_features_in_ == len(_fp), (
    f"pipeline expects {_pipe.n_features_in_} cols, frozen_primary has {len(_fp)}")
assert "specific_verb_ratio" not in _fp and "n_generic_verb_r100" not in _fp

# 2) prompt-hash assertion, re-run exactly as Block 1 does
_sha = hashlib.sha256(V2_PROMPT.encode()).hexdigest()[:16]
assert _sha == EXPECTED_PROMPT_SHA == _spec["prompt_sha256_16"] == "f52aae5d939d620e", \
    f"prompt SHA mismatch: {_sha} / {_spec['prompt_sha256_16']} / {EXPECTED_PROMPT_SHA}"

# 3) provenance + guard flags
assert _spec["test_extraction_model"] == MODEL_A == "qwen3.5:122b"
assert _spec["model_artifact"] == "pipe_v2_stable_surface.joblib"
assert _spec["dropped_degenerate"] == ["n_entities"]
assert _spec["unstable_features"] == ["n_entities", "n_generic_verb", "specific_verb_ratio"]
assert (FROZEN_DIR / "pipe_v2_rates_surface_deprecated.joblib").exists(), "audit copy missing"
assert not (FROZEN_DIR / "pipe_v2_rates_surface.joblib").exists(), "stale non-deprecated primary present"
assert _spec["test_set_used"] is False

print("consistency check - frozen spec + primary artifact")
print(f"  model_artifact        : {_spec['model_artifact']}")
print(f"  frozen_primary        : {len(_fp)} cols  == pipeline.n_features_in_ ({_pipe.n_features_in_})")
print(f"  excludes unstable     : specific_verb_ratio, n_generic_verb_r100  -> not present")
print(f"  prompt sha256[:16]    : {_sha}   (== EXPECTED_PROMPT_SHA, == spec)")
print(f"  test_extraction_model : {_spec['test_extraction_model']}")
print(f"  dropped_degenerate    : {_spec['dropped_degenerate']}")
print(f"  unstable_features     : {_spec['unstable_features']}")
print(f"  frozen_primary_cv_auc : {_spec['frozen_primary_cv_auc']:.4f}")
_pb = _spec["paired_bootstrap"]["frozen_primary_vs_surface"]
print(f"  paired dAUC (vs surf) : {_pb['delta_auc']:+.4f}  CI [{_pb['ci95'][0]:+.4f}, {_pb['ci95'][1]:+.4f}]"
      f"  P(<=0) {_pb['p_delta_le_0']:.3f}")
print(f"  deprecated artifact   : pipe_v2_rates_surface_deprecated.joblib  (kept for audit)")
print(f"  test_set_used         : {_spec['test_set_used']}")
print("\nALL CHECKS PASSED - pipeline frozen, test set untouched.")


consistency check - frozen spec + primary artifact
  model_artifact        : pipe_v2_stable_surface.joblib
  frozen_primary        : 20 cols  == pipeline.n_features_in_ (20)
  excludes unstable     : specific_verb_ratio, n_generic_verb_r100  -> not present
  prompt sha256[:16]    : f52aae5d939d620e   (== EXPECTED_PROMPT_SHA, == spec)
  test_extraction_model : qwen3.5:122b
  dropped_degenerate    : ['n_entities']
  unstable_features     : ['n_entities', 'n_generic_verb', 'specific_verb_ratio']
  frozen_primary_cv_auc : 0.8058
  paired dAUC (vs surf) : +0.0635  CI [+0.0065, +0.1197]  P(<=0) 0.016
  deprecated artifact   : pipe_v2_rates_surface_deprecated.joblib  (kept for audit)
  test_set_used         : False

ALL CHECKS PASSED - pipeline frozen, test set untouched.


## Block 11 — verdict

In [15]:
med_rho = stab.loc[COUNTS, "spearman"].median()
n_stab  = int(stab.loc[COUNTS + RATIOS, "stable"].sum()); n_tot = len(COUNTS) + len(RATIOS)
m, ci, pneg = paired("v2r_surf", "surf")

def auc_of(key): return roc_auc_score(y, PROBA[key])

print("Q1  Are the V2 features stable across LLMs?")
print(f"    median Spearman rho (counts) = {med_rho:.2f}")
print(f"    {n_stab}/{n_tot} count+ratio features stable at rho >= {STABILITY_MIN_RHO}")
print(f"    dropped as degenerate (zero_share > {ZERO_SHARE_MAX}): {DROP_COUNTS}")
print(f"    unstable: {list(stab.loc[COUNTS + RATIOS].query('not stable').index)}")
print()
print("Q2  Do V2 features add information beyond surface statistics?")
print(f"    surface alone      AUC = {auc_of('surf'):.3f}")
print(f"    V2 rates + surface AUC = {auc_of('v2r_surf'):.3f}")
print(f"    paired dAUC = {m:+.3f}   95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]   P(dAUC<=0) = {pneg:.3f}")
verdict = ("ADDS signal beyond surface" if ci[0] > 0 else
           "does NOT add demonstrable signal beyond surface (CI crosses 0)")
print(f"    -> {verdict}")
print()
print(f"    TF-IDF ceiling AUC = {auc_of('tfidf'):.3f}   (gap to V2+surface = "
      f"{auc_of('tfidf') - auc_of('v2r_surf'):+.3f})")
print("\nTEST SET: untouched. One evaluation, once, after this pipeline is frozen.")

Q1  Are the V2 features stable across LLMs?
    median Spearman rho (counts) = 0.68
    13/16 count+ratio features stable at rho >= 0.6
    dropped as degenerate (zero_share > 0.7): ['n_entities']
    unstable: ['n_entities', 'n_generic_verb', 'specific_verb_ratio']

Q2  Do V2 features add information beyond surface statistics?
    surface alone      AUC = 0.743
    V2 rates + surface AUC = 0.805
    paired dAUC = +0.063   95% CI [+0.004, +0.120]   P(dAUC<=0) = 0.019
    -> ADDS signal beyond surface

    TF-IDF ceiling AUC = 0.867   (gap to V2+surface = +0.062)

TEST SET: untouched. One evaluation, once, after this pipeline is frozen.
